In [1]:
!apt-get update
!apt-get install -y espeak-ng

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [2]:
!espeak-ng --version

eSpeak NG text-to-speech: 1.50  Data at: /usr/lib/x86_64-linux-gnu/espeak-ng-data


In [3]:
!pip install coqui-tts datasets soundfile huggingface_hub tqdm

In [4]:
import os
import numpy as np
import soundfile as sf
from tqdm import tqdm
from datasets import load_dataset, Dataset, Audio
from TTS.api import TTS
from huggingface_hub import login

In [5]:
HF_TOKEN = "<redacted>"
DATASET_NAME = "byteCode18/spoken-squad-memory-eval"
SPLIT = "test"

VERSION_NAME = "v2"
NOISE_SNR = 20

In [6]:
OUTPUT_DIR = f"./generated_audio_{VERSION_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [7]:
login(token=HF_TOKEN)

In [8]:
ds = load_dataset(DATASET_NAME, split=SPLIT)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [9]:
tts = TTS("tts_models/en/ljspeech/vits", progress_bar=False, gpu=True)

/usr/local/lib/python3.12/dist-packages/TTS/api.py:93: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


In [10]:
def add_noise(audio, snr_db):
    if snr_db is None:
        return audio

    signal_power = np.mean(audio ** 2)
    noise_power = signal_power / (10 ** (snr_db / 10))

    noise = np.random.normal(0, np.sqrt(noise_power), audio.shape)
    return audio + noise

In [11]:
#Generate Audio
audio_paths = []

for i, sample in tqdm(enumerate(ds), total=len(ds)):
    text = sample["instruction"]

    wav = tts.tts(text)

    if NOISE_SNR is not None:
        wav = add_noise(np.array(wav), NOISE_SNR)

    path = os.path.join(OUTPUT_DIR, f"sample_{i}.wav")
    sf.write(path, wav, 22050)

    audio_paths.append(path)

100%|██████████| 4704/4704 [14:34<00:00,  5.38it/s]


In [12]:
#Add Audio Column
new_column_name = f"instruction_{VERSION_NAME}"

ds = ds.add_column(new_column_name, audio_paths)

In [13]:
ds[0]

{'context': <datasets.features._torchcodec.AudioDecoder at 0x7b60243c52e0>,
 'instruction': 'What did Luther think was required to stop the violence?',
 'answer': 'personal presence',
 'duration_sec': 48.84,
 'instruction_len': 10,
 'answer_len': 2,
 'instruction_v2': './generated_audio_v2/sample_0.wav'}

In [14]:
ds = ds.cast_column(new_column_name, Audio())

In [15]:
ds[0]

{'context': <datasets.features._torchcodec.AudioDecoder at 0x7b60243c63c0>,
 'instruction': 'What did Luther think was required to stop the violence?',
 'answer': 'personal presence',
 'duration_sec': 48.84,
 'instruction_len': 10,
 'answer_len': 2,
 'instruction_v2': <datasets.features._torchcodec.AudioDecoder at 0x7b60243abef0>}

In [16]:
#Push to HuggingFace
ds.push_to_hub(
    f"byteCode18/spoken-squad-memory-eval-{VERSION_NAME}",
    private=False
)

Uploading the dataset shards:   0%|          | 0/20 [00:00<?, ? shards/s]

Map:   0%|          | 0/236 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   0%|          | 54.9kB /  190MB            

CommitInfo(commit_url='https://huggingface.co/datasets/byteCode18/spoken-squad-memory-eval-v2/commit/da0b4800ff660d39e2ab469223ab2aaa0091edb8', commit_message='Upload dataset', commit_description='', oid='da0b4800ff660d39e2ab469223ab2aaa0091edb8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/byteCode18/spoken-squad-memory-eval-v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='byteCode18/spoken-squad-memory-eval-v2'), pr_revision=None, pr_num=None)